# Tasks
Analysis of token input and output (when it is unrestrained)
Distribution of input and output tokens with Qwen Model Sizes
Distribution of input and output tokens with Gemma Model Sizes 
Analysis of failures vs. non-failure tasks on the token input/output count distributions
Analysis of thinking (reasoning) vs no thinking (non-reasoning) on the token input/output count distributions

Try the dataset with SWE-Bench

Plot Input/Output Token Distribution for Coding model across the qwen model family sizes with humaneval/SweBench
Plot Input/Output Token Distribution for Coding model across the gemma model family sizes with humaneval/SweBench
Plot input/output token distribution for coding model with non-reasoning 
Plot input/output token distribution for coding model with failure vs non failure


# TODO: write a function that takes in 4 directories of output folders and output the distribution of input and output tokens, need to parse each trial log to find token information from each llm output. Process the 4 output nothink folders for now and output the average input/output token length per turn

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def parse_token_usage_from_console_log(log_path: str) -> list[dict]:
    """Parse a console_log.txt file and extract token usage from each LLM OUTPUT.
    
    Returns a list of dicts, one per LLM call (turn), each with keys:
        prompt_tokens, completion_tokens, total_tokens, reasoning_tokens
    """
    with open(log_path, "r") as f:
        content = f.read()

    # Match CompletionUsage(...) blocks in LLM OUTPUT sections
    pattern = r"CompletionUsage\(completion_tokens=(\d+),\s*prompt_tokens=(\d+),\s*total_tokens=(\d+),\s*completion_tokens_details=\w+,\s*prompt_tokens_details=\w+,\s*reasoning_tokens=(\d+)\)"
    matches = re.findall(pattern, content)

    results = []
    for m in matches:
        results.append({
            "completion_tokens": int(m[0]),
            "prompt_tokens": int(m[1]),
            "total_tokens": int(m[2]),
            "reasoning_tokens": int(m[3]),
        })
    return results


def collect_token_data(output_dir: str) -> pd.DataFrame:
    """Collect token usage data from all trials in an output directory.
    
    Returns a DataFrame with columns:
        task, trial, turn, prompt_tokens, completion_tokens, total_tokens, reasoning_tokens
    """
    # Find the Results directory
    results_dir = os.path.join(output_dir, "Results")
    if not os.path.isdir(results_dir):
        raise FileNotFoundError(f"No Results directory in {output_dir}")

    # Find the scenario run directory (e.g., human_eval_AgentChat_20260220_...)
    scenario_runs = [d for d in os.listdir(results_dir) if os.path.isdir(os.path.join(results_dir, d))]
    if not scenario_runs:
        raise FileNotFoundError(f"No scenario run directories in {results_dir}")

    rows = []
    for scenario_run in scenario_runs:
        scenario_run_path = os.path.join(results_dir, scenario_run)
        # Find the scenario folder (e.g., human_eval_AgentChat)
        scenario_folders = [
            d for d in os.listdir(scenario_run_path)
            if os.path.isdir(os.path.join(scenario_run_path, d))
        ]
        for scenario_folder in scenario_folders:
            scenario_path = os.path.join(scenario_run_path, scenario_folder)
            # Iterate over task folders (e.g., HumanEval_0, HumanEval_1, ...)
            for task_name in sorted(os.listdir(scenario_path)):
                task_path = os.path.join(scenario_path, task_name)
                if not os.path.isdir(task_path):
                    continue
                # Iterate over trial folders (0, 1, 2, ...)
                for trial_name in sorted(os.listdir(task_path)):
                    trial_path = os.path.join(task_path, trial_name)
                    log_path = os.path.join(trial_path, "console_log.txt")
                    if not os.path.isfile(log_path):
                        continue
                    usages = parse_token_usage_from_console_log(log_path)
                    for turn_idx, usage in enumerate(usages):
                        rows.append({
                            "task": task_name,
                            "trial": int(trial_name),
                            "turn": turn_idx + 1,
                            **usage,
                        })

    return pd.DataFrame(rows)


def analyze_token_distributions(output_dirs: dict[str, str]) -> pd.DataFrame:
    """Analyze token distributions across multiple output directories.
    
    Args:
        output_dirs: dict mapping label -> output directory path
        
    Returns:
        Combined DataFrame with a 'model' column for the label.
    """
    all_dfs = []
    for label, path in output_dirs.items():
        print(f"Processing {label} from {path}...")
        df = collect_token_data(path)
        df["model"] = label
        all_dfs.append(df)
        print(f"  Found {len(df)} LLM calls across {df['task'].nunique()} tasks")

    combined = pd.concat(all_dfs, ignore_index=True)
    return combined

In [ ]:
# Process the 4 base nothink output folders
BASE_DIR = os.path.join(os.getcwd(), "..")

nothink_dirs = {
    "Qwen3-0.6B": os.path.join(BASE_DIR, "output_0.6b_nothink"),
    "Qwen3-1.7B": os.path.join(BASE_DIR, "output_1.7b_nothink"),
    "Qwen3-4B": os.path.join(BASE_DIR, "output_4b_nothink"),
    "Qwen3-8B": os.path.join(BASE_DIR, "output_8b_nothink"),
}

df = analyze_token_distributions(nothink_dirs)
print(f"\nTotal records: {len(df)}")
df.head()

In [ ]:
# Average input/output token length per turn for each model
avg_per_turn = (
    df.groupby(["model", "turn"])
    .agg(
        avg_prompt_tokens=("prompt_tokens", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        avg_total_tokens=("total_tokens", "mean"),
        count=("prompt_tokens", "count"),
    )
    .round(1)
)
print("Average Input/Output Tokens Per Turn (by model):")
print("=" * 80)
avg_per_turn

In [ ]:
# Overall average per model (across all turns)
overall_avg = (
    df.groupby("model")
    .agg(
        avg_prompt_tokens=("prompt_tokens", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        avg_total_tokens=("total_tokens", "mean"),
        total_turns=("prompt_tokens", "count"),
        num_tasks=("task", "nunique"),
    )
    .round(1)
)
print("Overall Average Input/Output Tokens Per Model:")
print("=" * 80)
overall_avg

In [ ]:
# Plot: Distribution of Input (Prompt) Tokens by Model
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

model_order = ["Qwen3-0.6B", "Qwen3-1.7B", "Qwen3-4B", "Qwen3-8B"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

# Input tokens distribution
ax = axes[0]
for model, color in zip(model_order, colors):
    subset = df[df["model"] == model]["prompt_tokens"]
    ax.hist(subset, bins=50, alpha=0.5, label=model, color=color, density=True)
ax.set_xlabel("Prompt Tokens (Input)")
ax.set_ylabel("Density")
ax.set_title("Distribution of Input Tokens per LLM Call")
ax.legend()

# Output tokens distribution
ax = axes[1]
for model, color in zip(model_order, colors):
    subset = df[df["model"] == model]["completion_tokens"]
    ax.hist(subset, bins=50, alpha=0.5, label=model, color=color, density=True)
ax.set_xlabel("Completion Tokens (Output)")
ax.set_ylabel("Density")
ax.set_title("Distribution of Output Tokens per LLM Call")
ax.legend()

plt.suptitle("Token Distribution Across Qwen3 Model Sizes (No-Think Mode)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Plot: Average Input/Output Tokens per Turn by Model (bar chart)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

avg_by_model = df.groupby("model").agg(
    avg_prompt=("prompt_tokens", "mean"),
    avg_completion=("completion_tokens", "mean"),
).reindex(model_order)

# Bar chart of averages
x = np.arange(len(model_order))
width = 0.35

ax = axes[0]
ax.bar(x - width/2, avg_by_model["avg_prompt"], width, label="Avg Prompt Tokens", color="#1f77b4")
ax.bar(x + width/2, avg_by_model["avg_completion"], width, label="Avg Completion Tokens", color="#ff7f0e")
ax.set_xticks(x)
ax.set_xticklabels(model_order)
ax.set_ylabel("Tokens")
ax.set_title("Average Input/Output Tokens per LLM Call")
ax.legend()
for i, (p, c) in enumerate(zip(avg_by_model["avg_prompt"], avg_by_model["avg_completion"])):
    ax.text(i - width/2, p + 10, f"{p:.0f}", ha="center", va="bottom", fontsize=8)
    ax.text(i + width/2, c + 10, f"{c:.0f}", ha="center", va="bottom", fontsize=8)

# Box plot comparison
ax = axes[1]
data_prompt = [df[df["model"] == m]["prompt_tokens"].values for m in model_order]
data_completion = [df[df["model"] == m]["completion_tokens"].values for m in model_order]

bp1 = ax.boxplot(data_prompt, positions=x - 0.2, widths=0.3, patch_artist=True,
                  boxprops=dict(facecolor="#1f77b4", alpha=0.5))
bp2 = ax.boxplot(data_completion, positions=x + 0.2, widths=0.3, patch_artist=True,
                  boxprops=dict(facecolor="#ff7f0e", alpha=0.5))
ax.set_xticks(x)
ax.set_xticklabels(model_order)
ax.set_ylabel("Tokens")
ax.set_title("Token Distribution per LLM Call (Box Plot)")
ax.legend([bp1["boxes"][0], bp2["boxes"][0]], ["Prompt Tokens", "Completion Tokens"])

plt.suptitle("Token Usage Across Qwen3 Model Sizes (No-Think Mode)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()